In [3]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pyarrow.feather as feather
from scipy.stats import kstest

#import importlib
import sys
import os
hostname = os.uname().nodename
if hostname == 'BlackBeast':
    path = '/home/hedvigs/snake_book/econ'
    site = 'home'
elif hostname == 'hedvig-hp-elitedesk-800-g5-twr':
    path = '/home/hedvigs/PycharmProjects/homewrs/plab_workflow/'
    site = 'work'
elif hostname == 'work-computer':
    path = '/mnt/work/workbench/hedvigs/snake_book/econ'
    site = 'server'
elif hostname == 'SilverFlex':
    path = '/home/hedvigs/gitrepos/plab_workflow/'
    site = 'silverFlex'

sys.path.append(path)
print(path)
from workflow.scripts.data_management import setup_data as gt


/home/hedvigs/gitrepos/plab_workflow/


In [106]:
def summarize_scores(
    subset=None, model=None, gen=None, fold=None, path=None, verbose=0
):
    inputs = [subset, gen, fold, model]
    config_names = ["subsets", "genomes", "folds", "models"]

    config_dict = {}

    for item, config_name in zip(inputs, config_names):
        if item is not None:
            config_dict[config_name] = [item]
        else:
            config_dict[config_name] = gt.read_config(config_name, path=path)
    y_pred = []
    missing = pd.DataFrame()
    k = 0

    for subset in config_dict["subsets"]:
        for model in config_dict["models"]:
            for gen in config_dict["genomes"]:
                for fold in config_dict["folds"]:
                    score_file = (
                        path
                        + f"results/scores/PTD/{subset}/{model}_{gen}_{fold}.csv"
                        )
                    score_file_adj = (
                        path
                        + f"results/scores/PTD/{subset}/{model}_{gen}_{fold}_adj.csv"
                        )
                    try:
                        try:
                            scores = pd.read_csv(score_file, sep="\t")
                        except FileNotFoundError:
                            scores = pd.read_csv(score_file_adj, sep='\t')
                            
                        scores["fold"] = fold
                        scores["gen"] = gen
                        scores["model"] = model
                        scores["subset"] = subset
                        y_pred.append(scores)
                    except FileNotFoundError:
                        
                        k += 1
                        missing.loc[k, "subset"] = subset
                        missing.loc[k, "model"] = model
                        missing.loc[k, "gen"] = gen
                        missing.loc[k, "fold"] = fold
    if k == 0:
        print(f" All files found!, {k} missing")
    elif verbose == 1:
        print(f"Files not found:", k)
    elif verbose == 2:
        print(f"Files not found:", k)
        subsetnames, subsetcounts = np.unique(missing["subset"], return_counts=True)
        for sname, scount in zip(subsetnames, subsetcounts):
            print(f"  {sname}: {scount} missing")
            missubset = missing[missing["subset"] == sname]
            modelnames, modelcounts = np.unique(missubset["model"], return_counts=True)
            for mname, mcount in zip(modelnames, modelcounts):
                print(f"    {mname}: {mcount} missing")
    elif verbose == 3:
        print("Total files not found:", k)
        for colname in missing.columns:
            names, counts = np.unique(missing[colname], return_counts=True)
            print(f"{colname}: ")
            for name, count in zip(names, counts):
                print(f"  {name}: {count} missing")
    elif verbose == 4:
        print(f"Files not found:", k)
        subsetnames, subsetcounts = np.unique(missing["subset"], return_counts=True)
        for sname, scount in zip(subsetnames, subsetcounts):
            print(f"  {sname}: {scount} missing")
            missubset = missing[missing["subset"] == sname]
            modelnames, modelcounts = np.unique(missubset["model"], return_counts=True)
            for mname, mcount in zip(modelnames, modelcounts):
                print(f"    {mname}: {mcount} missing")
                missmodel = missubset[missubset["model"] == mname]
                gennames, gencounts = np.unique(missmodel["gen"], return_counts=True)
                for gname, gcount in zip(gennames, gencounts):
                    print(f"        {gname}: {gcount} missing")

    full_set = pd.concat(y_pred)
    return full_set

In [107]:
subset = None #'top29' # wildcards["iSubset"]
model = None
gen = None  # wildcards["iGen"]

sum_df = summarize_scores(subset=subset, model=model, gen=gen, fold=None,path=path, verbose=4)
auc_df = sum_df.drop(columns=["number"]) # ,"fold", "train_score", "f1", "fbeta", "perm_score", "plr", "nlr", "pval_score", "auc_pred"])


Files not found: 15
  all: 15 missing
    knn: 3 missing
        combine: 1 missing
        f: 1 missing
        m: 1 missing
    lda: 2 missing
        combine: 1 missing
        m: 1 missing
    lrc: 3 missing
        combine: 1 missing
        f: 1 missing
        m: 1 missing
    nn: 2 missing
        f: 1 missing
        m: 1 missing
    qda: 2 missing
        m: 2 missing
    rfc: 2 missing
        combine: 1 missing
        m: 1 missing
    svc: 1 missing
        m: 1 missing


In [4]:
auc_df.auc_prob

0     0.56582
1     0.58531
2     0.58120
3     0.51667
4     0.58465
       ...   
15    0.54930
16    0.46684
17    0.55309
18    0.50000
19    0.54236
Name: auc_prob, Length: 8360, dtype: float64

In [108]:

sum_file= path + 'results/report/sum_file_adj.csv'
#sum_file= path + f'results/report/sum_file_26.csv'
auc_df.to_csv(sum_file, sep="\t", index=False)


## Adjust old score files to match new ones

In [53]:
subset = 'all'
gen=None
fold=None
model = None
inputs = [subset, gen, fold, model]
config_names = ["subsets", "genomes", "folds", "models"]
config_dict = {}
for item, config_name in zip(inputs, config_names):
    if item is not None:
        config_dict[config_name] = [item]
    else:
        config_dict[config_name] = gt.read_config(config_name, path=path)


In [104]:


for model in config_dict["models"]:
    for gen in config_dict["genomes"]:
        for fold in config_dict["folds"]:
            pruned_score_file = (
                path
                + f"results/scores/PTD/{subset}/{model}_{gen}_{fold}_pruned.csv"
                )
            try:
                pruned_score = pd.read_csv(pruned_score_file, sep="\t")
                new_score= pd.DataFrame()
                new_score['number'] = pruned_score.number
                new_score['test_perm'] = pruned_score.perm_score
                new_score['test_pval'] = pruned_score.pval_score
                new_score['test_score'] = pruned_score.auc_prob
                new_score['train_score'] = pruned_score.train_score
                new_score['auc_prob'] = pruned_score.auc_prob
                new_score['auc_pred'] = pruned_score.auc_pred
                new_score['r2'] = pruned_score.fbeta - 9
                npos,nneg = num_posneg(sums,fold)
                tn, fp, fn, tp = calc_confusion(pruned_score,npos, nneg)
               # print(pruned_score_file)
                new_score['tn'] = tn.astype(int)
                new_score['fp'] = fp.astype(int)
                new_score['fn'] = fn.astype(int)
                new_score['tp'] = tp.astype(int)
                new_score_file = (
                    path
                    + f"results/scores/PTD/{subset}/{model}_{gen}_{fold}_adj.csv"
                    )
                new_score.to_csv(new_score_file, sep='\t')
            except FileNotFoundError:
                continue
            


            
            


0    0.55239
1    0.55239
2    0.55239
3    0.55239
4    0.55239
5    0.55239
6    0.55239
7    0.55239
8    0.55239
9    0.55239
dtype: float64
0    0.413708
1    0.413708
2    0.413708
3    0.413708
4    0.413708
5    0.413708
6    0.413708
7    0.413708
8    0.413708
9    0.413708
dtype: float64
0    0.607378
1    0.607378
2    0.607378
3    0.607378
4    0.607378
5    0.607378
6    0.607378
7    0.607378
8    0.607378
9    0.607378
dtype: float64
0    0.583439
1    0.583439
2    0.583439
3    0.583439
4    0.583439
5    0.583439
6    0.583439
7    0.583439
8    0.583439
9    0.583439
dtype: float64
0    0.650294
1    0.650294
2    0.650294
3    0.650294
4    0.650294
5    0.650294
6    0.650294
7    0.650294
8    0.650294
9    0.650294
dtype: float64
0    0.70529
1    0.70529
2    0.70529
3    0.70529
4    0.70529
5    0.70529
6    0.70529
7    0.70529
8    0.70529
9    0.70529
dtype: float64
0    0.573362
1    0.573362
2    0.573362
3    0.573362
4    0.573362
5    0.573362
6    0

In [103]:
model='nn'
gen='f'
fold=4
pruned_score_file = (
    path
    + f"results/scores/PTD/{subset}/{model}_{gen}_{fold}_pruned.csv"
    )
pruned_score = pd.read_csv(pruned_score_file, sep="\t")
new_score= pd.DataFrame()
new_score['number'] = pruned_score.number
new_score['test_perm'] = pruned_score.perm_score
new_score['test_pval'] = pruned_score.pval_score
new_score['test_score'] = pruned_score.auc_prob
new_score['train_score'] = pruned_score.train_score
new_score['auc_prob'] = pruned_score.auc_prob
new_score['auc_pred'] = pruned_score.auc_pred
new_score['r2'] = pruned_score.fbeta - 9
npos,nneg = num_posneg(sums,fold)
tn, fp, fn, tp = calc_confusion(pruned_score,npos, nneg)
new_score['tn'] = tn.astype(int)
new_score['fp'] = fp.astype(int)
new_score['fn'] = fn.astype(int)
new_score['tp'] = tp.astype(int)
new_score

0    0.998742
1    0.000000
2    0.000000
3    0.000000
4    0.823664
5    0.000000
6    0.000000
7    0.605799
8    0.907747
9    0.000000
dtype: float64


,number,test_perm,test_pval,test_score,train_score,auc_prob,auc_pred,r2,tn,fp,fn,tp
0,299,0.50294,0.31683,0.51911,0.51513,0.51911,0.49937,-8.03910,3172,3,84,0
1,299,0.49860,0.99010,0.49969,0.44758,0.49969,0.50000,-8.03848,0,3176,0,84
2,299,0.50432,0.84158,0.51345,0.47886,0.51345,0.50000,-8.03848,0,3176,0,84
3,299,0.50303,0.51485,0.52294,0.50154,0.52294,0.50000,-8.03848,0,3176,0,84
4,299,0.49983,0.58416,0.56148,0.49459,0.56148,0.51303,-8.12870,2615,560,66,17
5,299,0.49747,0.90099,0.49971,0.46322,0.49971,0.50000,-8.03848,0,3176,0,84
6,299,0.50173,0.92079,0.50419,0.46598,0.50419,0.50000,-8.03848,0,3176,0,84
7,299,0.50175,0.91089,0.53121,0.47360,0.53121,0.54694,-8.26954,1924,1251,43,40
8,299,0.49792,0.71287,0.55071,0.48193,0.55071,0.51935,-8.08245,2883,292,73,10
9,299,0.49869,0.85149,0.50867,0.47137,0.50867,0.50000,-8.03848,0,3176,0,84


In [85]:
lka=calc_confusion(pruned_score,npos, nneg)

0    0.0
1    0.0
2    0.0
3    0.0
4    1.0
5    0.0
6    1.0
7    1.0
8    0.0
9    1.0
Name: plr, dtype: float64


In [15]:
sum_file= path + f'results/report/sum_file_26.csv'
sums = pd.read_csv(sum_file, sep='\t')

num_posneg(sums, 0)

(83, 3177)

In [14]:
def num_posneg(df, fold):
    df_fold = df[df.fold==fold]
    Npos = df_fold.fn + df_fold.tp
    Nneg = df_fold.tn + df_fold.fp
    return int(np.mean(Npos)), int(np.mean(Nneg))


In [102]:
def calc_confusion(pruned_df, npos, nneg):
    plr = np.maximum(pruned_df.plr.fillna(0), 0)
    nlr = np.maximum(pruned_df.nlr.fillna(0),0)
#    spec = (plr-1)/(plr-nlr)
    with np.errstate(
        divide="ignore", invalid="ignore"
        ):  # To handle division by zero gracefully
        spec = ((plr-1) / (plr - nlr)).where((plr - nlr) != 0, 0)
    sens = 1-nlr*spec
    print(spec)
    if np.any((sens<0)|(sens>1)|(0>spec)|(spec>1)):
        print(sens)
        raise ValueError("Spec/Sens invalid")
    tp = sens * npos
    fn = (1-sens)*npos
    tn = spec*nneg
    fp = (1-spec) * nneg
#    print(tp, fn, tn, fp)
    return tn, fp, fn, tp